In [ ]:
from dataloading import *
from bpetokenizer import *
from CJPEDataset import *
from model_arch import *

In [ ]:
train_ds, dev_ds, test_ds = load_cjpe()
tokenizer = load_bpe_tokenizer()

train_dataset = CJPEDataset(train_ds)
dev_dataset = CJPEDataset(dev_ds)
test_dataset = CJPEDataset(test_ds)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, collate_fn=collate_fn)

In [ ]:
class Phase4ClassificationModel(nn.Module):
    def __init__(self, base_model, num_classes=2, pad_token_id=0):
        super().__init__()
        self.base_model = base_model
        self.num_classes = num_classes
        self.pad_token_id = pad_token_id
        
        self.classification_head = nn.Linear(self.base_model.cfg.d_model, num_classes)

    def forward(self, idx):
        B, T = idx.shape

        x = self.base_model.tok_emb(idx) + self.base_model.pos_emb[:T]
        for block in self.base_model.blocks:
            x = block(x)
        hidden_states = self.base_model.ln_f(x)

        sequence_lengths = (idx != self.pad_token_id).sum(dim=1)
        last_token_indices = torch.clamp(sequence_lengths - 1, min=0)
        tail_output = hidden_states[torch.arange(B, device=idx.device), last_token_indices]
        logits = self.classification_head(tail_output)
        
        return logits

In [ ]:
import torch.optim

device = torch.device("mps" if torch.backends.mps.is_available() else "cuda" if torch.cuda.is_available() else "cpu")

teacher_base = DecoderLM(LMConfig(vocab_size=50260))
teacher_model = Phase4ClassificationModel(teacher_base, num_classes=2, pad_token_id=0)
teacher_model.load_state_dict(torch.load("teacher.pt", map_location=device))
teacher_model.to(device)
teacher_model.eval()
for param in teacher_model.parameters():
    param.requires_grad = False

student_base = DecoderLM(LMConfig(vocab_size=50260))
student_base.load_state_dict(torch.load("dec.pt", map_location=device))
keep_layers = [0, 2, 4, 6]
student_base.cfg.n_layers = len(keep_layers)
student_base.blocks = nn.ModuleList([student_base.blocks[i] for i in keep_layers])
student_model = Phase4ClassificationModel(student_base, num_classes=2, pad_token_id=0)
student_model.to(device)


optimizer = torch.optim.AdamW(student_model.parameters(), lr=5e-5)
criterion = nn.CrossEntropyLoss()

In [ ]:
from torch.nn.functional import F

def sequence_kd_loss(student_logits, teacher_logits, labels, temperature=2.0, alpha=0.5):
    ce_loss = F.cross_entropy(student_logits, labels)
    student_soft = F.log_softmax(student_logits / temperature, dim=-1)
    teacher_soft = F.softmax(teacher_logits / temperature, dim=-1)
    kl_loss = F.kl_div(student_soft, teacher_soft, reduction="batchmean") * (temperature ** 2)
    return (alpha * ce_loss) + ((1.0 - alpha) * kl_loss)

In [ ]:
from sklearn.metrics import f1_score

epochs = 3

for epoch in range(epochs):
    student_model.train()
    total_train_loss = 0
    
    for batch_idx, (b_inputs, b_labels) in enumerate(train_loader):
        b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
        
        optimizer.zero_grad()
        student_logits = student_model(b_inputs)
        with torch.no_grad():
            teacher_logits = teacher_model(b_inputs)
        loss = sequence_kd_loss(student_logits, teacher_logits, b_labels)
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
            
    student_model.eval()
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for b_inputs, b_labels in dev_loader:
            b_inputs, b_labels = b_inputs.to(device), b_labels.to(device)
            
            logits = student_model(b_inputs)
            preds = torch.argmax(logits, dim=1)
            
            all_preds.extend(preds.cpu().numpy())
            all_targets.extend(b_labels.cpu().numpy())
            
    macro_f1 = f1_score(all_targets, all_preds, average='macro')
    avg_train_loss = total_train_loss / len(train_loader)
    
    print("Train Loss:", avg_train_loss, "Val Macro-F1:", macro_f1)

    torch.save(student_model.state_dict(), "checkpoint" + str(epoch) + ".pt")